# Higher-order local fields and variable operators

This notebook solves small native examples and inspects the complete six-point
series archived in `examples/results/native-extensions`. The fields used as
references are explicit analytical expressions; these are not external solver
comparisons. Every spatial image shows the actual macrotriangulation. Profiles
preserve one-sided local values at macro interfaces.

The experiments cover primal Darcy P1–P4, Taylor–Hood and equal-order USFEM with
variable tensor resistance, conservative variable-coefficient RAD, Galerkin/SUPG
outflow layers, and P2/P3 transient heat. The gallery generator records all local
and trace degrees, subdivisions, quadrature orders, errors and field hashes.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
from IPython.display import SVG, display
from threadpoolctl import threadpool_limits

sys.path.insert(0, str(root / "examples"))
from manufactured import darcy_pressure, darcy_source, darcy_flux, stokes_velocity
from native_extension_data import (
    BoundaryLayer,
    rad_diffusion,
    rad_velocity,
    rad_reaction,
    rad_source,
    sine,
    heat_exact,
    heat_source,
    flow_source,
    resistance,
)
from pymhm import (
    TriangleMesh,
    FaceSpace,
    SkeletonSpace,
)

from functools import partial
from pymhm import Equation, MultiscaleProblem, assemble
from pymhm.fem.scalar.operators import boundary_data
from examples.formulations.flow import (
    VelocityPressureSpace, velocity_pressure_equations,
    tensor_residual_weight, velocity_pressure_fields,
)
from examples.formulations.scalar import (
    ScalarDiscretization, heat_problem, initial_coefficients, recover_scalar,
)
from examples.formulations.residual_transport import streamline_equations


In [ ]:
result_path = root / "examples/results/native-extensions"
report = json.loads((result_path / "report.json").read_text())
expected_counts = {"darcy": 24, "flow": 12, "rad": 12, "layer": 24, "heat": 12}
for family, count in expected_counts.items():
    assert len(report[family]) == count
    for row in report[family]:
        if "fields" in row:
            field = row["fields"]
            assert (
                hashlib.sha256((result_path / field["file"]).read_bytes()).hexdigest()
                == field["sha256"]
            )
print({family: len(report[family]) for family in expected_counts})


## P3 primal Darcy: pressure and its raw physical gradient

The pressure is `cos(pi*x)*cos(pi*y)`, the permeability is one, and the source is
`2*pi**2*p`. The full boundary carries its nonhomogeneous Dirichlet values.
P3 local pressure uses P2 skeletal moments. `flux_l2_error` evaluates
`-grad(p_h)`; this raw primal flux is not an H(div)-conforming reconstruction.


## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy

mesh = TriangleMesh.unit_square(1)
skeleton = SkeletonSpace(mesh, tuple((FaceSpace.uniform(2) for _ in mesh.faces)))
with threadpool_limits(limits=1):
    darcy_definition = define_darcy(
        mesh,
        degree=3,
        skeleton=skeleton,
        local_refinement=2,
        source=darcy_source,
        dirichlet=darcy_pressure,
        quadrature_order=8,
    )
    darcy_system = assemble(darcy_definition.problem)
    darcy = recover_darcy(
        darcy_definition,
        darcy_system,
        darcy_system.solve(constraints=pressure_constraints(darcy_definition, darcy_system)),
    )
errors = {
    "pressure_l2": darcy.l2_error(darcy_pressure, 10),
    "raw_flux_l2": darcy.flux_l2_error(darcy_flux, 10),
    "macro_balance": float(np.max(np.abs(darcy.conservation_residuals(10)))),
}
assert errors["macro_balance"] < 1e-11
errors


In [ ]:
display(SVG(filename=str(root / "docs/figures/high-order/darcy-convergence.svg")))
display(SVG(filename=str(root / "docs/figures/high-order/darcy-p4.svg")))

display(SVG(filename=str(root / "docs/figures/high-order/darcy-p4-flux.svg")))


## Variable tensor Brinkman resistance

The positive definite resistance is
`Gamma=(1+x+2*y)*array([[20,3],[3,1]])`. The force is the independently
differentiated Stokes force plus `Gamma @ u_exact`. Both components are coupled;
USFEM includes the complete tensor resistance, velocity Laplacian and pressure
gradient in its residual stabilization.

The six-point series holds 32 macrotriangles fixed, uses vector P1 traces and
`2*s` local subdivisions for TH P2/P1 and USFEM P2/P2. The third map panel is
`norm(u_h-u_exact)`, not a difference of magnitudes.


The application provider declares the grad-grad/divergence blocks and subtracts
$\tau(R(u,p),R(v,q))_T$, with
$R(u,p)=-\Delta u+\Gamma u+\nabla p$. The positive tensor residual weight is

$$
\tau_T=\frac{h_T^2}
 {\max(\Gamma_{\max,T}h_T^2,4/m_T)+4/m_T}.
$$

The existing pure inverse-bound owner computes $m_T$ from the physical P2
velocity gradients and Hessians; the sampled tensor bound and the assembly
quadrature match this smooth manufactured case. All source-residual terms are
included. `LocalEquations` declares B and C=-B.T; `Equation` supplies the
negative weak velocity boundary moments and pressure has its physical zero-mean gauge.


In [ ]:
flow_trace = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces), 2)
flow_space = VelocityPressureSpace(mesh, flow_trace, 2, 2, 2, 8)
provider = partial(
    velocity_pressure_equations, space=flow_space, source=flow_source,
    drag=resistance, residual_weight=tensor_residual_weight,
)
boundary, _ = boundary_data(flow_trace, (0.0, 0.0), order=8)
problem = MultiscaleProblem(
    Equation(0, -np.r_[boundary, np.zeros(2 * len(mesh.cells))]),
    provider, range(len(mesh.cells)), flow_trace.size, (2,) * len(mesh.cells),
)
with threadpool_limits(limits=1):
    system = assemble(problem)
    gauge = system.mean_constraint([record[2] for record in system.local_metadata], 0.0)
    coefficients = system.solve(constraints=(gauge,))
    brinkman = velocity_pressure_fields(system, coefficients, flow_space)
print("Small USFEM P2 velocity error:", brinkman.l2_error(stokes_velocity, 8))


In [ ]:
for method in ("taylor-hood", "usfem"):
    rows = [row for row in report["flow"] if row["method"] == method]
    print(method, [(row["segments"], row["velocity_l2"], row["pressure_l2"]) for row in rows])
display(SVG(filename=str(root / "docs/figures/high-order/flow-convergence.svg")))
display(SVG(filename=str(root / "docs/figures/high-order/tensor-brinkman.svg")))

display(SVG(filename=str(root / "docs/figures/high-order/tensor-brinkman-pressure.svg")))


## Variable conservative RAD and an independent small solve

The equation is `-div(K grad(u)) + div(beta*u) + c*u = f`, with
`K=.05*(1+x+y)*[[2,.3],[.3,1]]`, `beta=(1+x,.5+y)` and `c=.5+x*y`.
The analytic data supply the full conservative forcing for `sin(pi*x)*sin(pi*y)`.
The coefficient derivatives are explicit: `div(K)=(.115,.065)` and `div(beta)=2`.
They enter the full SUPG residual along with the polynomial Hessian.

The skeletal multiplier has the Robin convention
`(-K @ grad(u) + beta*u/2) @ n`; it is not the full conservative flux.


The declared SUPG bilinear and linear forms are

$$
\begin{aligned}
a_T(u,v)={}&(K\nabla u,\nabla v)_T+((c+\operatorname{div}\beta/2)u,v)_T\\
&+\tfrac12[(\beta\cdot\nabla u,v)_T-(u,\beta\cdot\nabla v)_T]
 +(\tau R(u),\beta\cdot\nabla v)_T,\\
R(u)={}&-\operatorname{div}(K\nabla u)+\beta\cdot\nabla u
 +(c+\operatorname{div}\beta)u,\\
L_T(v)={}&(f,v)_T+(\tau f,\beta\cdot\nabla v)_T.
\end{aligned}
$$

Analytical $\operatorname{div}K$ and $\operatorname{div}\beta$ are explicit
inputs; the positive streamline weight delegates to its existing pure owner.
The independent pairings are $B_T\lambda=\langle\lambda,v\rangle$ and
$C_Tu=-\langle u,\mu\rangle$. The multiplier remains the half-advection Robin
quantity. The small layer uses these same equations with constant coefficients.


In [ ]:
scalar_trace = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces))
rad_space = ScalarDiscretization(mesh, scalar_trace, rad_diffusion, 2, 2, 8)
provider = partial(
    streamline_equations, data=rad_space, velocity=rad_velocity,
    velocity_divergence=2.0, diffusion_divergence=(0.115, 0.065),
    reaction=rad_reaction, source=rad_source,
)
boundary, _ = boundary_data(scalar_trace, 0.0, order=8)
problem = MultiscaleProblem(
    Equation(0, -np.r_[boundary, np.zeros(len(mesh.cells))]),
    provider, range(len(mesh.cells)), scalar_trace.size, (1,) * len(mesh.cells),
)
with threadpool_limits(limits=1):
    system = assemble(problem)
    coefficients = system.solve()
    rad = recover_scalar(rad_space, coefficients, tuple(system.local_metadata))
print("SUPG P2 L2 error:", rad.l2_error(sine, 10))


In [ ]:
display(SVG(filename=str(root / "docs/figures/high-order/rad-convergence.svg")))
display(SVG(filename=str(root / "docs/figures/high-order/variable-rad.svg")))


## Outflow layers: distinguish error reduction from positivity

For `epsilon=.02` or `.005`, the exact field is
`x-(exp((x-1)/epsilon)-exp(-1/epsilon))/(1-exp(-1/epsilon))`.
It solves `-epsilon*Delta(u)+u_x=1`. Only the vertical edges have zero values;
the horizontal edges require the full nonzero exact boundary function.

The recorded Galerkin and SUPG P1 cases use 32 macrotriangles, P1 traces,
`4*s` local subdivisions, assembly order 20 and error order 24. Extrema over all
local nodal coefficients are the exact extrema of the broken P1 field. They
include weak boundary deviations. SUPG does not enforce a discrete maximum
principle. Vertical profile markers show every actual macroface crossing.


In [ ]:
layer_mesh = TriangleMesh.unit_square(2)
layer_trace = SkeletonSpace(layer_mesh, tuple(FaceSpace.uniform(1) for _ in layer_mesh.faces))
layer_exact = BoundaryLayer(0.005)
layer_space = ScalarDiscretization(layer_mesh, layer_trace, 0.005, 1, 4, 20)
provider = partial(
    streamline_equations, data=layer_space, velocity=(1.0, 0.0),
    velocity_divergence=0.0, diffusion_divergence=(0.0, 0.0), reaction=0.0, source=1.0,
)
boundary, _ = boundary_data(layer_trace, layer_exact.value, order=20)
problem = MultiscaleProblem(
    Equation(0, -np.r_[boundary, np.zeros(len(layer_mesh.cells))]),
    provider, range(len(layer_mesh.cells)), layer_trace.size, (1,) * len(layer_mesh.cells),
)
with threadpool_limits(limits=1):
    system = assemble(problem)
    layer = recover_scalar(layer_space, system.solve(), tuple(system.local_metadata))
print("Small SUPG layer error:", layer.l2_error(layer_exact.value, 24))


In [ ]:
for epsilon in (0.02, 0.005):
    exact = BoundaryLayer(epsilon)
    for method in ("galerkin", "supg"):
        rows = [row for row in report["layer"]
                if row["epsilon"] == epsilon and row["stabilization"] == method]
        print(epsilon, method,
              [(row["segments"], row["scalar_l2"], row["undershoot"], row["overshoot"])
               for row in rows])
    print("Exact maximum:", exact.maximum)
display(SVG(filename=str(root / "docs/figures/high-order/layer-errors.svg")))
display(SVG(filename=str(root / "docs/figures/high-order/layer-profiles.svg")))


## P2/P3 heat with a time-linear exact field

Set `u_exact=(1+t)*sin(pi*x)*sin(pi*y)` and
`f=(1+2*pi**2*(1+t))*sin(pi*x)*sin(pi*y)`. Its backward-Euler difference quotient
is exact in time, so the four-step archived curves primarily assess spatial
approximation and initial interpolation. They do not claim higher temporal order.


Each time step declares the local and global equations separately. The shared
scalar FEM kernel supplies stiffness A, mass M and source load f; the provider
writes

$$
\begin{aligned}
(A+M/\Delta t)u^{n+1}+B\lambda^{n+1}
 &=f^{n+1}+Mu^n/\Delta t,\\
C u^{n+1}&=g,\qquad C=-B^T.
\end{aligned}
$$

Initial nodal interpolation, retained constant moments, time nodes and quadrature
match the reference coefficient contract. The `heat_problem` helper returns
`MultiscaleProblem`; assembly, solving and field interpretation are explicit stages.


In [ ]:
heat_space = ScalarDiscretization(mesh, scalar_trace, 1.0, 2, 2, 8)
old = initial_coefficients(heat_space, sine)
with threadpool_limits(limits=1):
    for left, time in zip((0.0, 0.05), (0.05, 0.1), strict=True):
        problem = heat_problem(
            heat_space, old, time - left, source=partial(heat_source, time=time),
        )
        system = assemble(problem)
        coefficients = system.solve()
        heat = recover_scalar(heat_space, coefficients, tuple(system.local_metadata))
        old = coefficients.fields
print("Heat P2 error at t=.1:", heat.l2_error(lambda points: heat_exact(points, .1), 10))


In [ ]:
display(SVG(filename=str(root / "docs/figures/high-order/heat-convergence.svg")))
display(SVG(filename=str(root / "docs/figures/high-order/heat-p3.svg")))


## Recompute the full gallery

Run `pixi run -e notebooks gallery-native-extensions --workers 4` from the
repository. The optional `--sections` argument selects families; all completed
cases are checkpointed. `--reuse-results` redraws the archived data without PDE
solves. The numerical report records separate source hashes for each family and
hashes for every displayed field archive.
